# Temporary test notebook for pairs_trading/utils.py and pairs_trading/pairs_filter.py
Scratch file — safe to delete once satisfied.

In [7]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from pairs_trading.utils import find_pair_groups
from pairs_trading.pairs_filter import find_pairs, compare_pair, load_metadata

## find_pair_groups() — sector/industry bucketing

In [2]:
groups = find_pair_groups()

print('sectors found:', list(groups.keys()))
print()
print('Technology industries:', list(groups['Technology']['industries'].keys()))
print()
print('Semiconductors group:', groups['Technology']['industries']['Semiconductors'])
print()
print('Technology sector ETFs:', groups['Technology']['etfs'])

sectors found: ['Technology', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Healthcare', 'Financial Services', 'Energy', 'Industrials', 'Basic Materials', 'Utilities', 'Real Estate']

Technology industries: ['Semiconductors', 'Consumer Electronics', 'Software - Infrastructure', 'Communication Equipment', 'Semiconductor Equipment & Materials', 'Information Technology Services', 'Computer Hardware', 'Software - Application', 'Electronic Components', 'Scientific & Technical Instruments', 'Solar', 'Electronics & Computer Distribution']

Semiconductors group: ['NVDA', 'AVGO', 'MU', 'AMD', 'INTC', 'TXN', 'QCOM', 'ADI', 'MRVL', 'NXPI', 'MPWR', 'MCHP', 'ON', 'GFS', 'ALAB', 'CRDO', 'TSEM', 'MTSI', 'SITM', 'LSCC', 'RMBS', 'SMTC', 'SWKS', 'QRVO', 'CRUS', 'MXL', 'ALGM', 'SLAB', 'VSH', 'SYNA', 'DIOD', 'PI', 'NVTS', 'LASR', 'POWI', 'WOLF', 'POET', 'SKYT', 'AIP', 'AMBQ']

Technology sector ETFs: ['VGT', 'XLK', 'SMH', 'SOXX', 'IYW', 'FTEC', 'BAI', 'IGV']


In [3]:
print(groups['Technology']['industries'])

{'Semiconductors': ['NVDA', 'AVGO', 'MU', 'AMD', 'INTC', 'TXN', 'QCOM', 'ADI', 'MRVL', 'NXPI', 'MPWR', 'MCHP', 'ON', 'GFS', 'ALAB', 'CRDO', 'TSEM', 'MTSI', 'SITM', 'LSCC', 'RMBS', 'SMTC', 'SWKS', 'QRVO', 'CRUS', 'MXL', 'ALGM', 'SLAB', 'VSH', 'SYNA', 'DIOD', 'PI', 'NVTS', 'LASR', 'POWI', 'WOLF', 'POET', 'SKYT', 'AIP', 'AMBQ'], 'Consumer Electronics': ['AAPL', 'SONO'], 'Software - Infrastructure': ['MSFT', 'ORCL', 'PLTR', 'PANW', 'CRWD', 'SNPS', 'FTNT', 'NET', 'CRWV', 'XYZ', 'TWLO', 'ZS', 'VRSN', 'MDB', 'NTAP', 'CPAY', 'FFIV', 'AKAM', 'IOT', 'OKTA', 'GEN', 'DOCN', 'TOST', 'RBRK', 'CHKP', 'NTNX', 'GDDY', 'SAIL', 'CORZ', 'DOX', 'DBX', 'KLAR', 'S', 'PATH', 'BLSH', 'WEX', 'ZETA', 'NTSK', 'ACIW', 'RELY', 'GTLB', 'BOX', 'QLYS', 'DLO', 'CLBT', 'FOUR', 'VRNS', 'PAY', 'TDC', 'WIX', 'NN', 'NTCT', 'TENB', 'EEFT', 'CALX', 'STNE', 'PAGS', 'RAMP', 'INFQ', 'AVPT', 'ATEN', 'FLYW', 'BAND', 'MQ', 'FIVN', 'PAYO', 'APPN', 'EVTC', 'KDK', 'PICS', 'GCT', 'PRGS', 'AI'], 'Communication Equipment': ['CSCO', 'CIEN

In [4]:
# Sanity checks
assert 'Technology' in groups
assert 'Semiconductors' in groups['Technology']['industries']
assert 'NVDA' in groups['Technology']['industries']['Semiconductors']
assert 'XLK' in groups['Technology']['etfs']

all_tickers = {t for s in groups.values() for t in s['etfs']} | \
              {t for s in groups.values() for inds in s['industries'].values() for t in inds}
assert not any(t.endswith('USDT') for t in all_tickers), 'crypto leaked into groups'

print('find_pair_groups() checks passed —', len(all_tickers), 'tickers grouped across', len(groups), 'sectors')

find_pair_groups() checks passed — 2016 tickers grouped across 11 sectors


## pairs_filter — compare_pair() and find_pairs()

In [5]:
meta = load_metadata()

result = compare_pair('NVDA', 'AMD', meta)
print('NVDA vs AMD')
print(' structural :', result['structural_score'])
print(' description:', result['description_score'])
print(' combined   :', result['combined_score'])
print(' breakdown  :', result['breakdown'])

NVDA vs AMD
 structural : 1.0
 description: 0.1907
 combined   : 0.6763
 breakdown  : {'cross_class': False, 'industry_match': True, 'sector_match': True, 'industry': 'Semiconductors', 'sector': 'Technology'}


In [6]:
df = find_pairs('NVDA', meta, min_combined=0.3)
print(f'{len(df)} candidates for NVDA at min_combined=0.3')
df.head(10)

39 candidates for NVDA at min_combined=0.3


,symbol,asset_class,structural_score,description_score,combined_score,industry,sector,name
0,INTC,equities,1.0,0.2790,0.7116,Semiconductors,Technology,Intel Corporation
1,AVGO,equities,1.0,0.1934,0.6774,Semiconductors,Technology,Broadcom Inc.
2,AMD,equities,1.0,0.1722,0.6689,Semiconductors,Technology,"Advanced Micro Devices, Inc."
3,SITM,equities,1.0,0.1696,0.6678,Semiconductors,Technology,SiTime Corporation
4,MXL,equities,1.0,0.1341,0.6536,Semiconductors,Technology,"MaxLinear, Inc."
5,MU,equities,1.0,0.1187,0.6475,Semiconductors,Technology,"Micron Technology, Inc."
6,AMBQ,equities,1.0,0.1139,0.6455,Semiconductors,Technology,"Ambiq Micro, Inc."
7,QCOM,equities,1.0,0.1099,0.6440,Semiconductors,Technology,QUALCOMM Incorporated
8,QRVO,equities,1.0,0.1052,0.6421,Semiconductors,Technology,"Qorvo, Inc."
9,SMTC,equities,1.0,0.0958,0.6383,Semiconductors,Technology,Semtech Corporation


In [7]:
# Sanity checks
assert not df.empty, 'expected at least some candidates for NVDA'
assert (df['combined_score'] >= 0.3).all()
assert df['combined_score'].is_monotonic_decreasing, 'expected results sorted descending'
assert 'AMD' in df['symbol'].values

# Cross-class comparison (equity vs ETF) — should lean entirely on description score
cross = compare_pair('NVDA', 'XLK', meta)
assert cross['structural_score'] == 0.0, 'cross-class structural score should be 0'
assert cross['combined_score'] == cross['description_score'], 'cross-class combined should equal description score'
print('NVDA vs XLK (cross-class):', cross['structural_score'], cross['description_score'], cross['combined_score'])

print('\npairs_filter checks passed')

NVDA vs XLK (cross-class): 0.0 0.0 0.0

pairs_filter checks passed


## pair_tests — compute_correlation_prefilter() (Stage A, sourced from data/ohlcv.duckdb::massive_1min)

In [8]:
from pairs_trading.pair_tests import compute_correlation_prefilter

group = groups['Technology']['industries']['Semiconductors']
print(f'group ({len(group)}):', group)

result = compute_correlation_prefilter(group_tickers=group)
result.sort_values('latest_corr', ascending=False).head(10)

group (40): ['NVDA', 'AVGO', 'MU', 'AMD', 'INTC', 'TXN', 'QCOM', 'ADI', 'MRVL', 'NXPI', 'MPWR', 'MCHP', 'ON', 'GFS', 'ALAB', 'CRDO', 'TSEM', 'MTSI', 'SITM', 'LSCC', 'RMBS', 'SMTC', 'SWKS', 'QRVO', 'CRUS', 'MXL', 'ALGM', 'SLAB', 'VSH', 'SYNA', 'DIOD', 'PI', 'NVTS', 'LASR', 'POWI', 'WOLF', 'POET', 'SKYT', 'AIP', 'AMBQ']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AMBQ: only 248/541 trading days in window — excluded


,ticker_a,ticker_b,latest_corr,min_corr,std_corr,passed
605,SWKS,QRVO,0.905913,0.888538,0.007241,True
181,TXN,ADI,0.827135,0.813083,0.012589,True
111,AMD,INTC,0.821678,0.804209,0.006748,True
354,MPWR,DIOD,0.792032,0.781534,0.003772,True
186,TXN,ON,0.784849,0.779144,0.007842,True
557,LSCC,ALGM,0.784086,0.725603,0.024212,True
441,ALAB,CRDO,0.783596,0.744850,0.017202,True
492,TSEM,SMTC,0.782328,0.688760,0.036532,True
674,ALGM,AIP,0.774858,0.653126,0.045035,True
336,MPWR,ON,0.772959,0.772959,0.014139,True


In [9]:
# Sanity checks
n = len(group)
assert len(result) <= n * (n - 1) // 2
assert set(result.columns) == {'ticker_a', 'ticker_b', 'latest_corr', 'min_corr', 'std_corr', 'passed'}
assert result['passed'].dtype == bool
print(f'{len(result)} pairs, {result["passed"].sum()} passed at default thresholds')

# Group too small
small = compute_correlation_prefilter(group_tickers=group[:3])
assert small.empty
print('small-group edge case OK')

# Missing ticker
try:
    compute_correlation_prefilter(group_tickers=group + ['NOTATICKER'])
    assert False, 'expected ValueError'
except ValueError as e:
    assert 'NOTATICKER' in str(e)
    print('missing-ticker edge case OK:', e)

print('\npair_tests checks passed')

741 pairs, 44 passed at default thresholds
group_tickers has 3 members (<4) — skipping, nothing to pair.
small-group edge case OK


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

missing-ticker edge case OK: Tickers not found in massive_1min: ['NOTATICKER']

pair_tests checks passed


## pair_tests — compute_cointegration_test() (Stage B)

In [6]:
from pairs_trading.pair_tests import compute_cointegration_test

coint_result = compute_cointegration_test(result)
coint_result.sort_values('n_crossings', ascending=False).head(15)

ModuleNotFoundError: No module named 'pairs_trading'

In [11]:
# Sanity checks
n_stage_a_passed = result['passed'].sum()
assert len(coint_result) == n_stage_a_passed
assert set(coint_result.columns) == {'ticker_a', 'ticker_b', 'pvalue_ab', 'pvalue_ba', 'cointegrated', 'hedge_ratio', 'n_crossings', 'passed'}
assert coint_result['cointegrated'].dtype == bool
assert coint_result['passed'].dtype == bool

# passed implies cointegrated and enough crossings
passed_rows = coint_result[coint_result['passed']]
assert (passed_rows['cointegrated']).all()
assert (passed_rows['n_crossings'] > 6).all()

print(f'{len(coint_result)} Stage-A-passed pairs tested, {coint_result["cointegrated"].sum()} cointegrated, {coint_result["passed"].sum()} passed Stage B')

# Empty input edge case
import pandas as pd
empty_stage_a = pd.DataFrame(columns=['ticker_a','ticker_b','latest_corr','min_corr','std_corr','passed'])
empty_coint = compute_cointegration_test(empty_stage_a)
assert empty_coint.empty
print('empty-input edge case OK')

print('\ncompute_cointegration_test checks passed')

44 Stage-A-passed pairs tested, 8 cointegrated, 8 passed Stage B
empty-input edge case OK

compute_cointegration_test checks passed


## pair_tests — plot_pair_dashboard()

In [17]:
from pairs_trading.utils import plot_pair_dashboard

# Pick a pair that passed Stage B (cointegrated) to sanity-check the happy path
# best_pair = coint_result[coint_result['passed']].sort_values('n_crossings', ascending=False).iloc[0]
# print('Plotting:', best_pair['ticker_a'], 'vs', best_pair['ticker_b'])

fig = plot_pair_dashboard('ALAB','MTSI')
fig.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
# Sanity checks
import vectorbtpro as vbt
assert isinstance(fig, vbt.utils.figure.Figure)
assert len(fig.data) == 3  # ticker_a line, ticker_b line, crossings marker trace
trace_names = {t.name for t in fig.data}
assert trace_names == {best_pair['ticker_a'], best_pair['ticker_b'], 'crossings'}
crossings_trace = [t for t in fig.data if t.name == 'crossings'][0]
assert len(crossings_trace.x) > 0, 'expected at least one crossing marker for a passed pair'
assert 'PASS' in fig.layout.title.text  # both stage checks should read PASS for a Stage-B-passed pair

print('plot_pair_dashboard checks passed —', len(crossings_trace.x), 'chart crossings marked')
print(fig.layout.title.text)

plot_pair_dashboard checks passed — 10 chart crossings marked
MTSI vs LSCC  &nbsp;|&nbsp;  corr: latest=0.691 min=0.651 std=0.016 (PASS)&nbsp;|&nbsp;  coint: p_ab=0.015 p_ba=0.028 hedge_ratio=2.642 spread_crossings=64 (PASS)&nbsp;|&nbsp;  chart crossings=10
